<a href="https://colab.research.google.com/github/m1Febriansyah/244107020199-ML-12/blob/main/JS11Praktikum5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
from pathlib import Path
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import cv2
import numpy as np
import pandas as pd

# 1. Definisikan Lokasi Folder
train_dir = "images/training/"
test_dir = "images/test/"

# 2. Fungsi Load Dataset
def load_dataset(img_dir):
    p = Path(img_dir)
    dirs = p.glob('*')
    img_list = []
    for dir in dirs:
        label = str(dir).split('/')[-1]
        for file in dir.glob('*.jpg'):
            img = mpimg.imread(file)
            if img is not None:
                img_list.append((img, label))
    return img_list

# 3. Fungsi Preprocessing (Ubah ukuran & Encode Label)
def standarized_input(image):
    return cv2.resize(image, (1100,600))

def label_encoder(label):
    return 1 if label == 'day' else 0

def preprocess(img_list):
    std_img_list = []
    for item in img_list:
        std_img = standarized_input(item[0])
        img_label = label_encoder(item[1])
        std_img_list.append((std_img, img_label))
    return std_img_list

# 4. Fungsi Ekstraksi Fitur (Hitung Rata-rata Kecerahan)
def avg_brightness(image):
    img_hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    sum_brightness = np.sum(img_hsv[:,:,2])
    area = image.shape[0] * image.shape[1]
    return sum_brightness / area

# === EKSEKUSI DATA TRAINING & TESTING ===
print("Memproses data...")
train_img = load_dataset(train_dir)
test_img = load_dataset(test_dir)

train_std_img_list = preprocess(train_img)
test_std_img_list = preprocess(test_img)
print("Data siap digunakan!")

Memproses data...
Data siap digunakan!


In [17]:
def predict_label(img, threshold):
    avg = avg_brightness(img)
    return 1 if avg > threshold else 0

def evaluate(img_list, threshold):
    miss_labels = []
    for file in img_list:
        img, label = file[0], file[1]
        pred_label = predict_label(img, threshold)
        if pred_label != label:
            miss_labels.append((img, pred_label, label))

    total_img = len(img_list)
    corr_pred = total_img - len(miss_labels)
    accuracy = corr_pred / total_img
    print(f'Akurasi dengan Threshold {threshold}: {accuracy:.4f}')

print("--- Hasil Klasifikasi Manual ---")
print("Data Training:")
evaluate(train_std_img_list, threshold=120)
print("Data Testing:")
evaluate(test_std_img_list, threshold=120)

--- Hasil Klasifikasi Manual ---
Data Training:
Akurasi dengan Threshold 120: 0.8417
Data Testing:
Akurasi dengan Threshold 120: 0.8688


In [18]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Fungsi menyusun data kecerahan ke dalam Tabel (DataFrame)
def extract_avg_bright_feature(img_list):
    avg_list = [avg_brightness(img[0]) for img in img_list]
    labels = [img[1] for img in img_list]
    return pd.DataFrame(np.column_stack((avg_list, labels)), columns=['AVG_BRIGHT', 'LABELS'])

# Ekstraksi ke tabel
train_avg_img = extract_avg_bright_feature(train_std_img_list)
test_avg_img = extract_avg_bright_feature(test_std_img_list)

# Siapkan data X (Fitur) dan Y (Label)
X_train = train_avg_img.iloc[:,0].values.reshape(-1,1)
y_train = train_avg_img.iloc[:,1].astype('int')
X_test = test_avg_img.iloc[:,0].values.reshape(-1,1)
y_test = test_avg_img.iloc[:,1].astype('int')

# Latih Model SVM
model = SVC()
model.fit(X_train, y_train)

# Prediksi dan Evaluasi
acc_train = accuracy_score(y_train, model.predict(X_train))
acc_test = accuracy_score(y_test, model.predict(X_test))

print("--- Hasil Klasifikasi SVM ---")
print(f'Akurasi pada data train: {acc_train:.4f}')
print(f'Akurasi pada data test: {acc_test:.4f}')

--- Hasil Klasifikasi SVM ---
Akurasi pada data train: 0.8583
Akurasi pada data test: 0.9000
